# 01 · Solar PV Project Finance Model & Scenario Analysis

This notebook builds a **fully-integrated project finance model** for a utility-scale solar PV plant and then runs a **single-variable scenario sweep** to see how investor returns respond to the resource (sunlight) assumption.

The model is a classic *three-statement, cash-flow-waterfall* project finance model of the kind used to size non-recourse debt and price equity returns for an SPV (special purpose vehicle):

- **Revenues** — energy production (`MWh`) priced under a fixed-price **PPA** for the first years and at **merchant** market prices thereafter.
- **Costs & taxes** — opex, depreciation, financial expenses and corporate tax.
- **Financing** — bank debt sized off a target **DSCR** (Debt Service Coverage Ratio) and sculpted to the cash flow, with the residual funded by equity.
- **Outputs** — the three financial statements (P&L, Balance Sheet, Cash Flow) plus equity-return metrics: the **IRR** and **cash-on-cash** multiple.

The model is implemented with the [`finmodel`](https://github.com/) library, where each line item is a Python function decorated with `@row` and evaluated lazily per period — the spreadsheet-style cell references (`self.revenue(t-1)`) are resolved automatically, including the circular debt-sizing/interest loop via iterative calculation.

> **Roadmap.** This is the deterministic baseline. Notebook `02` builds the real-world solar resource dataset, and notebook `03` replaces the single-point sweep here with a full **Monte Carlo** simulation driven by a stochastic (Markov-chain) weather model.

In [46]:
from finmodel.model import Model, FormulaRow, SimpleRow, PredefinedFormats, PredefinedStyles, row, Format
from finmodel.scenarios import Scenarios
import pandas as pd
from pydantic import BaseModel
from datetime import datetime, timedelta
from dataclasses import dataclass, field, replace, asdict
from dateutil.relativedelta import relativedelta
import numpy as np
import numpy_financial as npf
import time

import plotly.graph_objects as go
import plotly.io as pio
from utils import plotly_setup

pio.templates.default = "theme_dark_1"

## 2 · Model inputs

All exogenous assumptions live in a single `Inputs` dataclass, which keeps the model logic clean and makes scenario analysis trivial (we just clone the dataclass with one field changed). The `__post_init__` derives the key project dates from durations: the construction period end, the **COD** (Commercial Operation Date, when the plant starts generating) and the end of the project's operating life.

The fields group into the usual project-finance buckets:

| Group | Fields | Meaning |
|---|---|---|
| **Timeline** | `construction_bop`, `construction_periods`, `project_life` | When the plant is built and how long it operates (here: 1y build + 25y life). |
| **Technical** | `installed_capacity`, `net_equivalent_hours`, `availability`, `degradation` | Production drivers. Energy ≈ capacity × equivalent full-load hours × availability × (1 − cumulative degradation). |
| **Revenue** | `ppa_*`, `merchant_price` | A fraction of output is sold under a fixed-price **PPA** for `ppa_duration`; the rest (and everything after the PPA) is sold at the **merchant** price. |
| **Cost** | `capex_per_mwp`, `opex_per_year` | Capital cost (€/MWp) and annual operating cost. |
| **Working capital** | `days_receivable`, `days_payable` | Timing of cash in/out vs. accrual. |
| **Debt** | `debt_tenor`, `debt_dscr`, `debt_interest_rate`, … | Loan terms; the loan is sized so cash flow covers debt service at the target DSCR. |
| **Macro** | `cpi_*`, `tax_rate` | Inflation indexation of prices/costs and corporate tax. |

## 1 · Setup

We import the `finmodel` modelling primitives (`Model`, the `@row` decorator, predefined `Format`s and `Style`s) plus the `Scenarios` helper for the sweep. `numpy_financial` (`npf`) provides the `irr` calculation, and Plotly is configured with a custom dark theme for the charts.

In [47]:
@dataclass
class Inputs:
    construction_bop: datetime
    construction_periods: relativedelta
    construction_eop: datetime = field(init=False)

    cod_bop: datetime = field(init=False)
    project_life: relativedelta
    cod_eop: datetime = field(init=False)

    installed_capacity: float
    net_equivalent_hours: float #np.ndarray
    availability: float
    degradation: float
    degradation_start: datetime

    ppa_production: float
    ppa_price: float
    ppa_duration: relativedelta

    merchant_price: float #np.ndarray

    capex_per_mwp: float
    opex_per_year: float

    days_receivable: float
    days_payable: float

    debt_tenor: relativedelta
    debt_max_leverage_over_capex: float
    debt_dscr: float
    debt_upfront_fee: float
    debt_interest_rate: float

    cpi_bop: datetime
    cpi_per_year: float
    tax_rate: float
    
    min_cash: float

    def __post_init__(self):
        self.construction_eop = self.construction_bop + self.construction_periods - timedelta(days=1)
        self.cod_bop = self.construction_eop + timedelta(days=1)
        self.cod_eop = self.cod_bop + self.project_life - timedelta(days=1)

In [48]:
format_3f = Format(formatter=lambda x: f'{x:.3f}')

### Base-case assumptions

The instance below is the **central scenario**: a 10 MW plant, ~1,500 net equivalent hours, €1.0 m/MWp capex, an 80% PPA at €40/MWh for 13 years with a €80/MWh merchant price thereafter, and a 15-year loan sized at a 1.25× DSCR. These are the numbers every line item in the model reads from.

In [49]:
inputs = Inputs(
    construction_bop=datetime(2023, 1, 1),
    construction_periods=relativedelta(years=1),
    project_life=relativedelta(years=25),
    installed_capacity=10,
    net_equivalent_hours=1800,
    availability=0.99,
    degradation=0.5/100,
    degradation_start=datetime(2026, 1, 1),
    ppa_production=0.8,
    ppa_duration=relativedelta(years=13),
    ppa_price=30.0,
    merchant_price=40.0,
    capex_per_mwp=600.e3,
    opex_per_year=100.e3,
    days_receivable=30.0,
    days_payable=90.0,
    debt_tenor=relativedelta(years=15),
    debt_max_leverage_over_capex=0.8,
    debt_interest_rate=0.03,
    debt_dscr=1.25,
    debt_upfront_fee=0.02,
    cpi_bop=datetime(2026, 1, 1),
    cpi_per_year=2./100,
    tax_rate=25./100,
    min_cash=1,
)

## 3 · The project finance model

`PVModel` defines the whole model as a class of `@row` formulas, one per line item. Each formula takes the period index `t` and may reference other rows at the same or earlier periods (e.g. `self.debt_eop(t-1)`), exactly like a spreadsheet. `finmodel` resolves the dependency graph and evaluates lazily. Rows are organised into the standard project-finance blocks:

- **Periods & Flags** — the time axis (annual periods) plus boolean switches that turn line items on/off: `construction`, `COD`, `ppa_period`, the cumulative `degradation_factor`, and the inflation accumulator `cpi_acc`.
- **Revenues** — production split into PPA vs. merchant volumes, priced and inflated, summing to `total_revenues`; then opex down to **EBITDA**, **EBIT**, **EBT** and **net income**.
- **Net Fixed Assets** — capex spent during construction and straight-line depreciation (`da`) over the operating life.
- **Balance Sheet** — fixed assets, receivables, cash, equity, payables and debt, with a `balance_check` row asserting assets = liabilities every period.
- **Cash Flow** — the **cash waterfall**: EBITDA → cash available for debt service → debt repayment → cash to shareholders → dividends.
- **Debt & Equity** — the financing logic. Debt is **sculpted**: each period's debt service is set so that `CFADS / DSCR` is available, and the loan drawdown equals the present sum of repayments. The equity plug funds whatever debt does not cover.

### Circular reference
This model is **circular**: financial expenses depend on the debt balance, which depends on debt sizing, which depends on cash flow, which depends on… financial expenses. `finmodel` breaks the loop with iterative calculation (see the next cell's `enable_iterative_calculation` flag) rather than algebraic substitution — the same way Excel's iterative-calc setting works.

In [50]:
class PVModel(Model):

    period_freq = relativedelta(years=1)

    # -----------------------------------------------
    # PERIODS
    # -----------------------------------------------

    @row(group="Periods", format=PredefinedFormats.DATE)
    def bop(self, t: int):
        if t == 0:
            return self.inputs.construction_bop
        
        return self.bop(t-1) + self.period_freq
    
    @row(group="Periods", format=PredefinedFormats.DATE)
    def eop(self, t: int):
        return self.bop(t) + self.period_freq - relativedelta(days=1)
    
    @row(group="Periods")
    def days(self, t: int):
        return (self.eop(t) - self.bop(t)).days + 1
    

    # -----------------------------------------------
    # FLAGS
    # -----------------------------------------------

    @row(group="Flags")
    def construction(self, t: int):
        return 1*(self.inputs.construction_bop <= self.bop(t) and self.inputs.construction_eop >= self.eop(t))
    
    @row(group="Flags")
    def COD(self, t: int):
        return 1*(self.inputs.cod_bop <= self.bop(t) and self.inputs.cod_eop >= self.eop(t))
    
    @row(group="Flags")
    def years_From_COD(self, t: int):
        if t == 0:
            return 0
        return (self.COD(t) + self.years_From_COD(t-1))*self.COD(t)
    
    @row(group="Flags", format=PredefinedFormats.PERCENTAGE)
    def degradation(self, t: int):
        return self.inputs.degradation
    

    @row(group="Flags", format=PredefinedFormats.PERCENTAGE)
    def degradation_factor(self, t: int):
        if t == 0:
            return 0
        if  self.bop(t) >= self.inputs.degradation_start:
            return self.degradation_factor(t-1)*(1-self.degradation(t))
        else:
            return 1.0
        

    @row(group="Flags")
    def ppa_period(self, t: int):
        return ((self.inputs.cod_bop + self.inputs.ppa_duration) >= self.eop(t))*self.COD(t)
    
    @row(group="Flags")
    def cpi_period(self, t: int):
        return self.bop(t) >= self.inputs.cpi_bop
    
    @row(group="Flags", format=format_3f)
    def cpi_acc(self, t: int):
        if t == 0:
            return 1.0
        return self.cpi_acc(t-1)*(1 + self.inputs.cpi_per_year*self.cpi_period(t))


    # -----------------------------------------------
    # REVENUES
    # -----------------------------------------------


    @row(group="Revenues")
    def installed_capacity(self, t: int):
        return self.inputs.installed_capacity * self.COD(t)
    
    @row(group="Revenues")
    def net_equivalent_hours(self, t: int):
        return self.inputs.net_equivalent_hours * self.COD(t)
    
    @row(group="Revenues", format=PredefinedFormats.PERCENTAGE)
    def availability(self, t: int):
        return self.inputs.availability * self.COD(t)
    
    @row(group="Revenues")
    def production(self, t: int):
        return self.installed_capacity(t)*self.net_equivalent_hours(t)*self.availability(t)*self.degradation_factor(t)
    
    @row(group="Revenues")
    def production_ppa(self, t: int):
        return self.production(t)*self.ppa_period(t)*self.inputs.ppa_production
    
    @row(group="Revenues")
    def production_merchant(self, t: int):
        return self.production(t) - self.production_ppa(t)
    
    @row(group="Revenues")
    def price_ppa(self, t: int):
        return self.inputs.ppa_price * self.COD(t) * self.cpi_acc(t)
    
    @row(group="Revenues")
    def price_merchant(self, t: int):
        return self.inputs.merchant_price * self.COD(t)
    
    @row(group="Revenues")
    def revenues_ppa(self, t: int):
        return self.price_ppa(t) * self.production_ppa(t)/1e3
    
    @row(group="Revenues")
    def revenues_merchant(self, t: int):
        return self.price_merchant(t) * self.production_merchant(t)/1e3
    

    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def total_revenues(self, t: int):
        return self.revenues_ppa(t) + self.revenues_merchant(t)
    

    # -----------------------------------------------
    # NET FIXED ASSETS
    # -----------------------------------------------
    

    @row(group="Net Fixed Assets")
    def capex(self, t: int):
        construction_periods = sum(self.construction(t_) for t_ in range(self.periods))
        return self.construction(t)*(self.inputs.capex_per_mwp*self.inputs.installed_capacity/1e3)/construction_periods
    

    @row(group="Net Fixed Assets")
    def da(self, t: int):
        total_capex = sum(self.capex(t_) for t_ in range(self.periods))
        total_cod = sum(self.COD(t_) for t_ in range(self.periods))

        return -self.COD(t)*(total_capex/total_cod)
    
    @row(group="Net Fixed Assets")
    def fixed_assets_bop(self, t: int):
        if t == 0:
            return 0
        return self.fixed_assets_eop(t-1)
    
    @row(group="Net Fixed Assets")
    def fixed_assets_eop(self, t: int):
        return self.fixed_assets_bop(t) + self.da(t) + self.capex(t)
    

    # -----------------------------------------------
    # P&L
    # -----------------------------------------------

    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def revenues(self, t: int):
        return self.total_revenues(t)
    
    @row(group="Revenues")
    def opex(self, t: int):
        return self.COD(t) * (-self.inputs.opex_per_year/1e3)* self.cpi_acc(t)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def ebitda(self, t: int):
        return self.revenues(t) + self.opex(t)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def ebit(self, t: int):
        return self.ebitda(t) + self.da(t)


    @row(group="Revenues",)
    def net_financial_expenses(self, t: int):
        return self.financial_expenses(t)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def ebt(self, t: int):
        return self.ebit(t) + self.net_financial_expenses(t)
    
    @row(group="Revenues")
    def taxes(self, t: int):
        return self.ebt(t) * (-self.inputs.tax_rate)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def net_income(self, t: int):
        return self.ebt(t) + self.taxes(t)
    

    # -----------------------------------------------
    # Balance
    # -----------------------------------------------

    @row(group="Balance")
    def net_fixed_assets(self, t: int):
        return self.fixed_assets_eop(t)
    
    @row(group="Balance")
    def receivable(self, t: int):
        return self.revenues(t)*self.inputs.days_receivable/self.days(t) 

    @row(group="Balance")
    def cash(self, t: int):
        return self.cash_eop(t)

    @row(group="Balance", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def total_assets(self, t: int):
        return  self.net_fixed_assets(t) + self.receivable(t) + self.cash(t)
    

    @row(group="Balance")
    def equity(self, t: int):
        return self.equity_eop(t)
    
    @row(group="Balance")
    def payables(self, t: int):
        return -1*self.opex(t)*self.inputs.days_payable/self.days(t)


    @row(group="Balance")
    def debt(self, t: int):
        return self.debt_eop(t)
    
    @row(group="Balance", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def total_liabilities(self, t: int):
        return  self.payables(t) + self.equity(t) + self.debt(t)

    
    @row(group="Balance", format=PredefinedFormats.BOOLEAN)
    def balance_check(self, t: int):
        return abs(self.total_assets(t) - self.total_liabilities(t)) < 0.1

    # -----------------------------------------------
    # Cash Flow
    # -----------------------------------------------

    @row(group="Cash Flow")
    def working_capital(self, t: int):
        if t == 0:
            return 0
        return (self.receivable(t-1) - self.payables(t-1)) - (self.receivable(t) - self.payables(t))
    
    @row(group="Cash Flow")
    def cf_to_debt_service(self, t: int):
        return self.ebitda(t) - self.capex(t) + self.working_capital(t) + self.taxes(t)
    
    @row(group="Cash Flow")
    def cf_to_debt_repayment(self, t: int):
        return self.cf_to_debt_service(t) + self.net_financial_expenses(t)
    
    @row(group="Cash Flow")
    def cf_to_min_cash(self, t: int):

        cf_available_to_min_cash = self.cf_to_debt_repayment(t) + self.debt_repayment(t) + self.debt_drawdown(t)

        if cf_available_to_min_cash < 0:
            return 0
        elif self.cash_bop(t) < self.inputs.min_cash:
            return -min(self.inputs.min_cash - self.cash_bop(t), cf_available_to_min_cash)
        else:
            return 0 
    

    @row(group="Cash Flow")
    def cf_to_shareholders(self, t: int):
        return self.cf_to_debt_repayment(t) + self.debt_repayment(t) + self.debt_drawdown(t) + self.cf_to_min_cash(t)
            

    @row(group="Cash Flow")
    def cash_variation(self, t: int):
        return self.cf_to_shareholders(t) + self.capital_increase(t) + self.dividends(t) - self.cf_to_min_cash(t)

    @row(group="Cash Flow")
    def cash_bop(self, t: int):
        if t == 0:
            return 0
        return self.cash_eop(t-1)
    
    @row(group="Cash Flow")
    def cash_eop(self, t: int):
        return self.cash_bop(t) + self.cash_variation(t)
    
    # -----------------------------------------------
    # Debt
    # -----------------------------------------------

    @row(group="Debt")
    def tenor_debt_flag(self, t: int):
        return self.COD(t)*(t <= self.inputs.debt_tenor.years)
    

    @row()
    def debt_bop(self, t: int):
        if t == 0:
            return 0
        return self.debt_eop(t-1)
    
    @row(group="Debt")
    def debt_drawdown(self, t: int):
        total_debt = sum(self.debt_repayment(t_) for t_ in range(self.periods))
        if t==0:
            return -total_debt
        else:
            return 0
    
    @row()
    def debt_service(self, t: int):
        return self.tenor_debt_flag(t)*self.cf_to_debt_service(t)/self.inputs.debt_dscr

    @row()
    def debt_repayment(self, t: int):
        return -(self.debt_service(t) + self.financial_expenses(t)) * self.tenor_debt_flag(t)
    
    @row()
    def debt_eop(self, t: int):
        return  self.debt_bop(t) + self.debt_repayment(t) + self.debt_drawdown(t)
    

    @row()
    def debt_average(self, t: int):
        return self.debt_bop(t) + self.debt_drawdown(t)/2 + self.debt_repayment(t)*(1/4)

    @row()
    def financial_expenses(self, t: int):
        return self.debt_average(t)*(-self.inputs.debt_interest_rate)


    @row(group="Equity")
    def total_uses(self, t: int):
        return self.capex(t)
    
    @row(group="Equity")
    def total_sources(self, t: int):
        return self.total_uses(t)


    @row(group="Equity")
    def equity_bop(self, t: int):
        if t == 0:
            return 0
        return self.equity_eop(t-1)
    
    
    @row(group="Equity")
    def capital_increase(self, t: int):
        total_debt = sum(self.debt_repayment(t_) for t_ in range(self.periods))
        return (self.total_sources(t) + total_debt)*self.construction(t)
    
    @row(group="Equity")
    def dividends(self, t: int):
        return -max(0, min(self.equity_bop(t), self.cf_to_shareholders(t)) )
    
    @row(group="Equity")
    def net_income_to_equity(self, t: int):
        return self.net_income(t)
    
    @row(group="Equity")
    def equity_eop(self, t: int):
        return self.equity_bop(t) + self.capital_increase(t) + self.net_income_to_equity(t) + self.dividends(t)

    
    @row(group="Ratios", format=PredefinedFormats.PERCENTAGE)
    def debt_over_capex(self, t: int):
        if t == 0:
            total_debt = sum(self.debt_repayment(t_) for t_ in range(self.periods))
            total_capex = sum(self.capex(t_) for t_ in range(self.periods))
            return -total_debt/total_capex
        else:
            return 0



## 4 · Build & solve

We instantiate the model over **26 annual periods** (1 construction year + 25 operating years) and call `calculate()`. Because of the circular debt/interest loop, `enable_iterative_calculation=True` is required: the solver re-evaluates the model repeatedly until values stop changing.

> **Note on convergence.** `damping=1.0` and `max_iterations=100` were tuned for *this* model — the default damping of 0.5 over-damps the iteration and prevents it from settling. `model.show()` renders the full styled model as a financial-statement table.

In [51]:
model = PVModel(
    periods=26,
    inputs=inputs,
    style=PredefinedStyles.JETBRAINS_DARK_THEME,
    enable_iterative_calculation=True,
    max_iterations=100,   # was 10; with damping=1.0 the model converges by iter 8-9
    damping=1.0,        # was default 0.5, which over-damps and prevents convergence here
)


model.calculate()
model.show()

## 5 · Scenario analysis — resource sensitivity

The single most uncertain input for a solar project is the **resource**: how much sun the site actually receives, captured here by `net_equivalent_hours`. We use the `Scenarios` helper to sweep it across a realistic range and measure the impact on equity returns.

Two output metrics are defined as functions of a solved model:

- **`coc` (cash-on-cash)** — total dividends returned to shareholders divided by total capital invested (a simple money-multiple).
- **`irr`** — the internal rate of return of the equity cash-flow stream (capital calls out, dividends in), computed with `numpy_financial.irr`.

We then build 100 input variants spanning **900 → 2,000 equivalent hours** and solve the full model for each. This is a *deterministic* one-at-a-time sensitivity — every other assumption is held at base case. (Notebook `03` generalises this into a probabilistic Monte Carlo over thousands of correlated weather/price paths.)

In [52]:
scenarios = Scenarios(model)

def output_coc(model:PVModel):
    dividends = -model.get_result_data('dividends')
    capital_invested =  model.get_result_data('capital_increase')
    return float(np.sum(dividends)/np.sum(capital_invested))
    
def output_irr(model:PVModel):
    dividends = -model.get_result_data('dividends')
    capital_invested =  model.get_result_data('capital_increase')
    returns = dividends - capital_invested
    return float(npf.irr(returns))

scenarios.add_output('coc', output_coc)
scenarios.add_output('irr', output_irr)

scenarios_inputs = [replace(inputs, net_equivalent_hours=x) for x in np.linspace(1000, 2000, 100)]

scenarios.run_scenarios(scenarios_inputs)
df = scenarios.get_scenarios_df()

Calculating scenarios: 100%|██████████| 100/100 [00:07<00:00, 14.15it/s]


In [53]:
df.head()

input                                                       \
  construction_bop     construction_periods construction_eop    cod_bop   
0       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
1       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
2       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
3       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
4       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   

                                                           \
               project_life    cod_eop installed_capacity   
0  relativedelta(years=+25) 2048-12-31                 10   
1  relativedelta(years=+25) 2048-12-31                 10   
2  relativedelta(years=+25) 2048-12-31                 10   
3  relativedelta(years=+25) 2048-12-31                 10   
4  relativedelta(years=+25) 2048-12-31                 10   

                                                 ...  \
  net_equivalent_hours availability degradation  ...   
0           1000.00000         0.99       0.005  ...   
1           1010.10101         0.99       0.005  ...   
2           1020.20202         0.99       0.005  ...   
3           1030.30303         0.99       0.005  ...   
4           1040.40404         0.99       0.005  ...   

                                                                              \
  debt_max_leverage_over_capex debt_dscr debt_upfront_fee debt_interest_rate   
0                          0.8      1.25             0.02               0.03   
1                          0.8      1.25             0.02               0.03   
2                          0.8      1.25             0.02               0.03   
3                          0.8      1.25             0.02               0.03   
4                          0.8      1.25             0.02               0.03   

                                               output            
     cpi_bop cpi_per_year tax_rate min_cash       coc       irr  
0 2026-01-01         0.02     0.25        1  0.788411 -0.013299  
1 2026-01-01         0.02     0.25        1  0.803365 -0.012264  
2 2026-01-01         0.02     0.25        1  0.818527 -0.011232  
3 2026-01-01         0.02     0.25        1  0.833901 -0.010202  
4 2026-01-01         0.02     0.25        1  0.849491 -0.009174  

[5 rows x 30 columns]

## 6 · Results

The scenario DataFrame holds every input column alongside the computed `coc` and `irr` outputs. Plotting **equity IRR against net equivalent hours** shows the project's operating leverage: revenue scales almost linearly with the resource, but a large fixed cost base (capex + debt service) means IRR is highly geared to it — a modest swing in sunshine produces an outsized swing in returns. This is exactly the sensitivity the Monte Carlo model in notebook `03` quantifies as a full probability distribution.

In [54]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x= df[('input', 'net_equivalent_hours')],
    y= df[('output', 'irr')],
    
))

fig.update_layout(
    title="Returns",
    xaxis=dict(
        title="Net Equivalent Hours",
        tickformat=".0f"
    ),
    yaxis=dict(
        title="IRR",
        tickformat=".1%"
    )
)

fig.show()